In [10]:
import numpy as np
#import multiprocessing
import math
import pickle
import codecs

from tqdm import tqdm
from rdflib import Graph, URIRef, BNode, Literal, plugin
from rdflib.namespace import RDF, RDFS, OWL,Namespace, NamespaceManager
from rdflib.serializer import Serializer
import ifcopenshell
from ifcopenshell import geom

In [7]:
# ここを任意のパスに変えてください
ifc_path = './files/IfcOpenHouse.ifc'
# ifc_path = 'files/TKY-080c_INTEC_Interior_A22.ifc'
#ifc_path = 'files/I-REF_with_Room.ifc'
#ifc_path = 'files/TKY-080_Exterior model_A22.ifc'

rdf_path = "./output/{}.ttl".format(ifc_path.split('/')[-1].split('.')[0])

# IfcOpenShellの設定
try:
    ifc_file = ifcopenshell.open(ifc_path)
except:
    print(ifcopenshell.get_log())
else:
    # https://blenderbim.org/docs-python/ifcopenshell/geometry_settings.html
    settings = geom.settings()
    

./output/IfcOpenHouse.ttl


In [12]:
site = 'R90'

host = 'http://takenaka.co.jp/futaba/{0}/'.format(site)

device = host + 'device#'
building = host + 'building#'
floor = host + 'storey#'
space = host + 'space#'

bot = 'https://w3id.org/bot#'
rdfs_a = 'http://www.w3.org/2000/01/rdf-schema#type'
iot = 'http://iotschema.org/'
mozzila = 'https://iot.mozilla.org/schemas/'
brick = 'https://brickschema.org/ontology/1.3'

# 注：ifcOWLではないことに注意．以下の名前空間は独自定義
ifc = 'http://www.buildingsmart-tech.org/IFC2x3#'

g = Graph()
host_ns = Namespace(host)
device_ns = Namespace(device)
building_ns = Namespace(building)
floor_ns = Namespace(floor)
space_ns = Namespace(space)
bot_ns = Namespace(bot)
ifc_ns = Namespace(ifc)


namespace_manager = NamespaceManager(Graph())
namespace_manager.bind('inst', host_ns, override=False)
namespace_manager.bind('bot', bot_ns, override=False)
namespace_manager.bind('ifc', ifc_ns, override=False)
namespace_manager.bind('device', device_ns, override=False)
namespace_manager.bind('building', building_ns, override=False)
namespace_manager.bind('storey', floor_ns, override=False)
namespace_manager.bind('space', space_ns, override=False)

g.namespace_manager = namespace_manager


In [9]:
def format_ifc_info(info) -> dict:
    """ IfcOpenShellの情報はmultiprocessingを含んでいるので一般オブジェクトに変換する
    
    Args:
        info (_type_): IfcEntityのメタデータ

    Returns:
        dict: _description_
    """
    ret = dict()
    for key in info.keys():
        item = info.get(key)
        if type(item) != ifcopenshell.entity_instance and item != None:
            ret[key] = item
    return ret


def get_project_info(ifc_file, settings, name='Sample') :
    """ プロジェクトの情報を取得する

    Args:
        ifc_file (_type_): _description_
        settings (_type_): _description_
        name (str, optional): _description_. Defaults to 'Sample'.

    Returns:
        str, str, str: _description_
    """
    import math
    # IfcProjectから名前をとる
    prj = ifc_file.by_type('IfcProject')[0]
    name_ = prj.LongName if prj.LongName != 'プロジェクト名' else name
    name_ = name if name_ is None else name_
    
    # IfcSiteから緯度経度を取得
    site = ifc_file.by_type('IfcSite')[0]
    lat = '.'.join([str(i) for i in site.RefLatitude]) if site.RefLatitude is not None else ""
    lon = '.'.join([str(i) for i in site.RefLongitude]) if site.RefLongitude is not None else ""
 
    return name_, lat, lon


def get_properties(element):
    """ IFCのオブジェクトよりプロパティを抽出

    Args:
        element (_type_): _description_

    Returns:
        dict : プロパティセット
    """
    ret = vars(element)

    # propertyがない場合はエラーになる
    if hasattr(element, 'IsDefinedBy'):
        for tmp in element.IsDefinedBy:
            if tmp.is_a('IfcRelDefinesByProperties'):            
                pset = tmp.RelatingPropertyDefinition
                if pset.is_a('IfcPropertySet'):
                    for prop in pset.HasProperties:
                        try:
                            name = prop.Name
                            val = prop.NominalValue
                            ret[name] = val.wrappedValue
                        except:
                            print('Invalid Property: ' + name)
                elif pset.is_a('IfcElementQuantity'):
                    # 面積の場合がある
                    name = pset.Name
                    measurement = pset.MethodOfMeasurement 
                    quantities = pset.Quantities[0] 
                    #from IPython.core.debugger import Pdb; Pdb().set_trace()
                    if quantities.is_a('IfcQuantityArea'):
                        label = quantities.Name.replace(' ', '_')
                        description = quantities.Description
                        unit = quantities.Unit
                        # 記法の変更
                        area_value = quantities.AreaValue
                        ret[label]  = area_value
                else:
                    print('Invalid Type: ' + vars(pset)['type'])
            elif tmp.is_a('IfcRelDefinesByType'):
                #TODO クラス（Family）の定義
                pass
            else:
                print('Invalid Type : ' + vars(tmp)['type'])
      
    # 不要なプロパティの刈込(主観で決めている)
    ret.pop('OwnerHistory', None)
    ret.pop('CompositionType', None)
    ret.pop('Representation', None)
    ret.pop('ObjectPlacement', None)
    # ObjectTypeとほぼ等価
    ret.pop('Reference', None)
    # その他
    addr = ret.pop('BuildingAddress', None)
    if addr:
        addr = addr.AddressLines[0]
        ret['Address'] = addr
    return ret


def create_resource(entity, obj, name_=None, ns_=None, ADD_ID=True):
    props = get_properties(entity)
    name =  get_unicode(props['Name']) 
    ns = ns_ if ns_ is not None else host_ns
    inst_name = "_".join([name.split(":")[0].replace(" ", "_"), str(props['id'])]) if ADD_ID else name.split(":")[0]
    inst_name =  inst_name if name_ is None else name_
    resource = ns.term(inst_name)
    add_literal(resource, props)
    g.add((resource, RDF.type, obj))
    return resource


def add_literal(resource, props):
    for label in props:
        if props[label]:
            name = get_unicode(label).strip().replace(' ', '').replace('　', '')
            value = get_unicode(props[label])
           # print(name, value)
            g.add((resource, ifc_ns.term(name), Literal(value)))


def get_unicode(val):
    return codecs.decode(val, 'unicode-escape') if '\\u' in str(val) else val

In [14]:
# SITEは1つと仮定する。
# 名前を書き換える
site = ifc_file.by_type("IfcSite")[0]
site_resource = create_resource(site, bot_ns.Site, 'R90')

# Building
building = site.IsDecomposedBy[0].RelatedObjects[0]
building_resource = create_resource(building, bot_ns.Building, 'research', building_ns)

# リンク生成
g.add((site_resource, bot_ns.hasBuilding, building_resource))

AttributeError: 'NoneType' object has no attribute 'split'

In [ ]:
# TODO 構成システム類の抽出（空調等のシステムを表すIfcSystem)
systems = building.ServicedBySystems
print(len(systems))
storeys = building.IsDecomposedBy[0].RelatedObjects
storeys